# Week 8 Exercise: Multi-Agent Deal Hunting System

A self-contained demo of the "Price is Right" architecture from Week 8. Uses **mock agents** so it runs without Modal, HuggingFace, or external APIs.

**Architecture:**
- **ScannerAgent** → Scans for deals (mock RSS)
- **EnsembleAgent** → Estimates fair market price (mock LLM/RAG)
- **MessagingAgent** → Sends alerts for good deals
- **PlanningAgent** → Orchestrates: scan → price → rank → alert

**Flow:** Planning Agent runs a cycle: Scanner finds deals → Ensemble prices each → best deal (largest discount) triggers alert.

In [ ]:
# Install dependencies
!pip install -q gradio pydantic

In [ ]:
# Imports
import os
import json
import logging
import time
from typing import List, Optional

import gradio as gr
from pydantic import BaseModel

## Data Models

In [ ]:
class Deal(BaseModel):
    product_description: str
    price: float
    url: str


class DealSelection(BaseModel):
    deals: List[Deal]


class Opportunity(BaseModel):
    deal: Deal
    estimate: float
    discount: float

## Mock Agents

In [ ]:
class MockAgent:
    name = "MockAgent"

    def log(self, message: str):
        logging.info(f"[{self.name}] {message}")


class MockScannerAgent(MockAgent):
    name = "ScannerAgent"

    def scan(self, memory=None) -> DealSelection:
        self.log("Scanning RSS feeds for deals...")
        time.sleep(0.5)
        deals = [
            Deal(
                product_description="Apple iPad Pro 11-inch 256GB WiFi - M2 chip, Liquid Retina display, Face ID.",
                price=749.99,
                url="https://example.com/ipad",
            ),
            Deal(
                product_description="Sony WH-1000XM5 Wireless Noise Cancelling Headphones - 30hr battery.",
                price=329.99,
                url="https://example.com/sony",
            ),
            Deal(
                product_description="Logitech MX Master 3S Wireless Mouse - Ergonomic, 8K DPI.",
                price=69.99,
                url="https://example.com/logitech",
            ),
        ]
        self.log(f"Found {len(deals)} deals")
        return DealSelection(deals=deals)


class MockEnsembleAgent(MockAgent):
    name = "EnsembleAgent"

    def price(self, description: str) -> float:
        self.log("Estimating fair market price...")
        time.sleep(0.3)
        if "iPad" in description:
            return 899.0
        elif "Sony" in description:
            return 398.0
        elif "Logitech" in description:
            return 99.0
        return 150.0


class MockMessagingAgent(MockAgent):
    name = "MessagingAgent"

    def alert(self, opportunity: Opportunity):
        self.log(f"Alert: ${opportunity.discount:.2f} discount on {opportunity.deal.product_description[:40]}...")


class MockPlanningAgent(MockAgent):
    name = "PlanningAgent"
    DEAL_THRESHOLD = 50

    def __init__(self):
        self.scanner = MockScannerAgent()
        self.ensemble = MockEnsembleAgent()
        self.messenger = MockMessagingAgent()

    def plan(self, memory=None) -> Optional[Opportunity]:
        if memory is None:
            memory = []
        self.log("Starting planning cycle")
        selection = self.scanner.scan(memory)
        if not selection or not selection.deals:
            return None
        opportunities = []
        for deal in selection.deals:
            estimate = self.ensemble.price(deal.product_description)
            discount = estimate - deal.price
            opportunities.append(Opportunity(deal=deal, estimate=estimate, discount=discount))
        opportunities.sort(key=lambda x: x.discount, reverse=True)
        best = opportunities[0]
        self.log(f"Best deal: ${best.discount:.2f} discount")
        if best.discount >= self.DEAL_THRESHOLD:
            self.messenger.alert(best)
        return best

## Deal Agent Framework

In [ ]:
class MockDealAgentFramework:
    MEMORY_FILE = "week8_mock_memory.json"

    def __init__(self):
        self.memory: List[Opportunity] = self._read_memory()
        self.planner: Optional[MockPlanningAgent] = None

    def _read_memory(self) -> List[Opportunity]:
        if os.path.exists(self.MEMORY_FILE):
            try:
                with open(self.MEMORY_FILE, "r") as f:
                    data = json.load(f)
                return [Opportunity(**item) for item in data]
            except Exception:
                return []
        return []

    def _write_memory(self):
        data = [opp.model_dump() for opp in self.memory]
        with open(self.MEMORY_FILE, "w") as f:
            json.dump(data, f, indent=2)

    def init_agents(self):
        if not self.planner:
            self.planner = MockPlanningAgent()
            logging.info("Agent framework initialized")

    def run(self) -> List[Opportunity]:
        self.init_agents()
        result = self.planner.plan(memory=self.memory)
        if result:
            self.memory.append(result)
            self._write_memory()
        return self.memory

## Gradio UI

In [ ]:
# Configure logging to capture agent output
logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(message)s", datefmt="%H:%M:%S")

framework = MockDealAgentFramework()


def opps_to_table(opps: List[Opportunity]) -> List[List]:
    return [
        [o.deal.product_description, f"${o.deal.price:.2f}", f"${o.estimate:.2f}", f"${o.discount:.2f}", o.deal.url]
        for o in opps
    ]


def scan_for_deals():
    opps = framework.run()
    return opps_to_table(opps)


def build_ui():
    with gr.Blocks(title="The Price is Right - Mock") as ui:
        gr.Markdown("## The Price is Right - Multi-Agent Deal Hunting (Mock)")
        gr.Markdown("Click **Scan for Deals** to run one planning cycle. The framework scans mock deals, estimates prices, and surfaces the best opportunity.")
        table = gr.Dataframe(
            headers=["Description", "Price", "Estimate", "Discount", "URL"],
            wrap=True,
            col_count=5,
            row_count=10,
            value=opps_to_table(framework.memory),
        )
        btn = gr.Button("Scan for Deals", variant="primary")
        btn.click(scan_for_deals, outputs=table)
    return ui

In [ ]:
# Launch the UI
build_ui().launch()